# 🛡️ Bug Bounty Copilot - Autonomous Training Pipeline (Kaggle T4 GPU)

This notebook trains an advanced offensive security reasoning model using **Unsloth (QLoRA 4-bit)** on **Qwen2.5-7B-Instruct**.
- **Dataset**: Curated PentesterLand Bug Bounty Chain-of-Thought (CoT) pairs.
- **Hardware**: Kaggle Free T4 GPU (Background Execution Supported).
- **Output**: LoRA adapter saved to `/kaggle/working/bugbounty_copilot_lora`.

In [ ]:
# 1. Install Unsloth & Core Dependencies (Optimized for T4 GPU)
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Check for Previous Accumulated LoRA Weights (Mount & Resume)
from unsloth import FastLanguageModel
import torch, os, glob

max_seq_length = 4096
dtype = None # Auto detection (Float16 for T4)
load_in_4bit = True

# Search for mounted previous kernel weights in /kaggle/input/
possible_loras = glob.glob('/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors', recursive=True)
if not possible_loras:
    possible_loras = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

loaded_from_previous = False
if possible_loras:
    prev_adapter_dir = os.path.dirname(possible_loras[0])
    print(f'[✓] FOUND PREVIOUS DAY ACCUMULATED WEIGHTS at: {prev_adapter_dir}')
    print('[*] Resuming and stacking incremental training on top of previous day progress...')
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = prev_adapter_dir,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )
        FastLanguageModel.for_training(model)
        loaded_from_previous = True
        print('[✓] Successfully loaded previous day accumulated weights! Knowledge is compounding.')
    except Exception as e:
        print(f'[!] Could not load previous weights directly ({e}). Falling back to base model.')

if not loaded_from_previous:
    model_name = 'Qwen/Qwen2.5-7B-Instruct'
    print(f'[*] Initializing fresh base model: {model_name} in 4-bit...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                          'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = 'none',
        use_gradient_checkpointing = 'unsloth',
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    print('[✓] Attached fresh QLoRA adapters to base model!')


In [ ]:
# 3. Verify Active LoRA Training Status
FastLanguageModel.for_training(model)
print('[✓] Model & LoRA Adapters verified in active training mode!')


In [ ]:
# 4. Load & Format Training Dataset (ChatML Format)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen2.5 standard
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Load our verified Elite Bug Bounty Dataset (1,500 PentesterLand CoT pairs)
dataset_url = "https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/elite_bugbounty_cot.jsonl"
print(f"[*] Loading verified dataset from: {dataset_url}")
dataset = load_dataset("json", data_files=dataset_url, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"[✓] Loaded {len(dataset)} elite offensive security reasoning samples!")

In [ ]:
# 5. Configure SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/checkpoints",
        save_steps = 100,
    ),
)
print("[✓] Trainer configured and ready!")

In [ ]:
# 6. Train the Model!
print("[*] Starting training loop on GPU...")
trainer_stats = trainer.train()
print("[✓] Training completed successfully!")

In [ ]:
# 7. Test Inference (Sample Vulnerability Reasoning)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an Elite Offensive Security & Bug Bounty AI Copilot."},
    {"role": "user", "content": "Target scope: api.target.com/v1/user/profile. How would you methodically test for IDOR and privilege escalation?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
# 8. Save LoRA Adapters to Kaggle Output
output_dir = "/kaggle/working/bugbounty_copilot_lora"
print(f"[*] Saving trained LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("[✓] LoRA adapters permanently saved to output! Available in Kaggle output tab.")